# Advanced: Multi-Strategy Fake Phone Number Generation with PAMOLA.CORE

**Goal**: Master all 3 phone generation strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Multi-country generation (US, UK, CA...)
- **Strategy 2**: Custom operator codes with dictionaries
- **Strategy 3**: Format-aware generation (E.164, local, international)
- Advanced consistency mechanisms (PRGN vs mapping)
- Multi-field phone processing
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of PRGN consistency
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── fake_data/
        └── 02_fake_phone_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Import required modules
try:
    from pamola_core.fake_data.operations.phone_op import FakePhoneOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Restart your Python kernel/notebook")
    print("   2. Verify: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 8 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique counts)
- Country code and phone type distribution

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'phone_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Country codes for variety
    countries = ['1', '44', '49', '86']  # US, UK, Germany, China
    country_weights = [0.5, 0.2, 0.15, 0.15]  # US dominant
    
    # Phone types
    phone_types = ['mobile', 'landline', 'voip']
    type_weights = [0.6, 0.3, 0.1]
    
    # Generate phone numbers based on country
    def generate_phone(country_code):
        if country_code == '1':  # US
            area = np.random.choice(['555', '212', '310', '415', '713'])
            return f"+1 ({area}) {np.random.randint(100,999)}-{np.random.randint(1000,9999)}"
        elif country_code == '44':  # UK
            return f"+44 {np.random.randint(20,79)} {np.random.randint(1000,9999)} {np.random.randint(1000,9999)}"
        elif country_code == '49':  # Germany
            return f"+49 {np.random.randint(30,89)} {np.random.randint(10000000,99999999)}"
        else:  # China
            return f"+86 {np.random.randint(130,189)} {np.random.randint(10000000,99999999)}"
    
    country_codes = np.random.choice(countries, n_records, p=country_weights)
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'phone_primary': [generate_phone(cc) for cc in country_codes],
        'phone_mobile': [generate_phone(np.random.choice(countries, p=country_weights)) for _ in range(n_records)],
        'phone_work': [generate_phone(np.random.choice(countries, p=country_weights)) for _ in range(n_records)],
        'country_code': country_codes,
        'phone_type': np.random.choice(phone_types, n_records, p=type_weights),
        'region': np.random.choice(['us', 'eu', 'ca'], n_records, p=[0.5, 0.3, 0.2]),
        'verified': np.random.choice([True, False], n_records, p=[0.8, 0.2])
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): varies")

print(f"\n📞 Phone Distribution:")
print(f"  Country Codes: {df['country_code'].value_counts().to_dict()}")
print(f"  Phone Types: {df['phone_type'].value_counts().to_dict()}")
print(f"  Regions: {df['region'].value_counts().to_dict()}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Prepare Custom Operator Codes Dictionary

**How to use:**
- Run to check/create custom operator codes dictionary
- Uses existing file if found
- Creates example dictionary if missing

**What you'll see:**
- File status (✓ found or 🔧 created)
- Dictionary structure summary
- Operator counts per country
- Sample operator codes

**Dictionary structure:**
- Country codes mapped to operator prefixes
- US: 555, 212, 310, 415, etc.
- UK: 20, 121, 131, etc.
- Germany: 30, 40, 89, etc.

**Note:** Edit file to match your operator codes before processing

In [ ]:
# Define operator codes path (in examples directory)
examples_dir = project_root / 'examples'
operator_path = examples_dir / 'data_examples' / 'operator_codes.json'

# Check if file exists
if operator_path.exists():
    print(f"✓ Found existing operator codes: {operator_path}")
    print(f"📂 Using existing file for Strategy 2\n")
    
    with open(operator_path, 'r') as f:
        existing_codes = json.load(f)
    
    print("📊 Existing Dictionary Info:")
    print(f"  Format: v{existing_codes.get('format_version', 'N/A')}")
    print(f"  Countries: {len(existing_codes.get('operator_codes', {}))}")
    
    print("\n💡 To use different codes, replace or rename the file.")
else:
    print("⚠️  Operator codes not found!")
    print("🔧 Creating example operator codes dictionary...\n")
    
    operator_codes = {
        "format_version": "1.0",
        "description": "Operator/area codes for phone number generation",
        "operator_codes": {
            "1": {  # US
                "country": "United States",
                "codes": ["212", "310", "415", "555", "617", "713", "312", "202", "206", "305"]
            },
            "44": {  # UK
                "country": "United Kingdom",
                "codes": ["20", "121", "131", "141", "151", "161"]
            },
            "49": {  # Germany
                "country": "Germany",
                "codes": ["30", "40", "69", "89", "211", "221"]
            },
            "86": {  # China
                "country": "China",
                "codes": ["10", "20", "21", "25", "27", "28"]
            }
        }
    }
    
    try:
        operator_path.parent.mkdir(parents=True, exist_ok=True)
        with open(operator_path, 'w') as f:
            json.dump(operator_codes, f, indent=2)
        print(f"✓ Example operator codes created: {operator_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {operator_path} (file may be open)")

    print(f"\n📊 Statistics:")
    print(f"  Format version: {operator_codes['format_version']}")
    print(f"  Countries: {len(operator_codes['operator_codes'])}")
    for cc, data in operator_codes['operator_codes'].items():
        print(f"    +{cc} ({data['country']}): {len(data['codes'])} codes")

    print("\n💡 Edit this file to customize operator codes!")

## Step 4: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `"strategy1_field": "phone_primary"` (multi-country)
   - `"strategy2_field": "phone_mobile"` (custom operators)
   - `"strategy3_field": "phone_work"` (format-aware)
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique phone counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "phone_primary",   # Multi-country generation
    "strategy2_field": "phone_mobile",    # Custom operator codes
    "strategy3_field": "phone_work"       # Format-aware generation
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_phone_001",
    task_type="multi_strategy_phone",
    description="Multi-strategy fake phone generation",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Multi-Country Generation

**How to use:**
- Uses country_code field to generate appropriate phone numbers
- Automatically adapts formats (US, UK, Germany, China)
- Run to execute multi-country strategy

**Key parameters:**
- `default_country='us'`: Default country
- `country_code_field='country_code'`: Uses dataset's country column
- `preserve_country_code=True`: Maintains original country codes
- `preserve_operator_code=False`: Generates new operator codes
- `mode='ENRICH'`: Adds new column, keeps original

**What you'll see:**
- Configuration summary
- Progress bar: validation → generation → metrics → viz → save
- Completion time (3-5 seconds for 1000 records)
- Multi-country phone numbers

**Note:** Creates `phone_primary_multicountry` with country-appropriate formats

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: MULTI-COUNTRY GENERATION")
print("=" * 80 + "\n")

tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Multi-country",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s1 = FakePhoneOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy1_field']}_multicountry",
    output_format='csv',
    
    # Phone generation
    default_country='us',
    country_code_field='country_code',
    preserve_country_code=True,
    preserve_operator_code=False,
    format=None,  # Auto-detect
    region=None,
    
    # Validation
    validate_source=True,
    handle_invalid_phone='generate_new',
    
    # Consistency
    consistency_mechanism='prgn',
    key='multi-country-2025',
    context_salt='country-phones',
    
    # Metrics
    detailed_metrics=True,
    max_retries=3,
    
    # Processing
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Country handling: Uses '{operation_s1.country_code_field}' field")
print(f"  Preserve country code: {operation_s1.preserve_country_code}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_multicountry',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_multicountry' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_multicountry"
    
    print(f"\n📊 Sample Multi-Country Phones:")
    sample_by_country = result_df_s1.groupby('country_code').head(2)
    for _, row in sample_by_country.iterrows():
        print(
            f"  [+{str(row['country_code']):2s}] "
            f"{str(row[field_s1]):<30} → {row[output_field_s1]}"
        )

## STRATEGY 2: Custom Operator Codes

**How to use:**
- Uses custom operator codes dictionary from Step 3
- Generates phones with specific area/operator codes
- Run to execute custom operator strategy

**Key parameters:**
- `operator_codes_dict`: Path to custom codes dictionary
- `preserve_country_code=True`: Keep original codes
- `preserve_operator_code=True`: Use dictionary operators
- `consistency_mechanism='mapping'`: Deterministic mapping
- `save_mapping=True`: Saves phone mappings

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → dict load → generation → metrics → viz → save
- Completion time (3-5 seconds)
- Phones with custom operator codes
- Mapping file saved for reproducibility

**Note:** Creates `phone_mobile_custom` with dictionary-based operators

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: CUSTOM OPERATOR CODES")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Custom operators",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = FakePhoneOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy2_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy2_field']}_custom",
    output_format='csv',
    
    # Phone generation with custom operators
    default_country='us',
    country_code_field='country_code',
    operator_codes_dict=str(operator_path),
    preserve_country_code=True,
    preserve_operator_code=True,  # Use dictionary codes
    format=None,
    
    # Validation
    validate_source=True,
    handle_invalid_phone='generate_new',
    
    # Consistency with mapping
    consistency_mechanism='mapping',
    save_mapping=True,
    
    # Metrics
    detailed_metrics=True,
    max_retries=3,
    
    # Processing
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Operator dictionary: {operator_path.name}")
print(f"  Consistency: {operation_s2.consistency_mechanism}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_custom',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results (NEWEST file)
output_files_s2 = sorted(
    list((task_dir / 'strategy2_custom' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_custom"
    
    print(f"\n📊 Sample Custom Operator Phones:")
    for _, row in result_df_s2.head(10).iterrows():
        print(f"  {row[output_field_s2]}")

## STRATEGY 3: Format-Aware Generation

**How to use:**
- Generates phones in specific formats (E.164, local, international)
- Format preservation with regional adaptation
- Run to execute format-aware strategy

**Key parameters:**
- `format='E164'`: Generate in E.164 format (+1234567890)
- `preserve_country_code=True`: Keep original codes
- `region='us'`: Regional formatting rules
- `consistency_mechanism='prgn'`: PRGN consistency

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → format setup → generation → metrics → viz → save
- Completion time (3-5 seconds)
- Consistently formatted phone numbers

**Note:** Creates `phone_work_e164` with standardized E.164 format

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: FORMAT-AWARE GENERATION")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Format-aware",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = FakePhoneOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy3_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy3_field']}_e164",
    output_format='csv',
    
    # Phone generation with specific format
    default_country='us',
    country_code_field='country_code',
    format='E164',  # Standardized format
    region='us',
    preserve_country_code=True,
    preserve_operator_code=False,
    
    # Validation
    validate_source=True,
    handle_invalid_phone='generate_new',
    
    # Consistency
    consistency_mechanism='prgn',
    key='format-e164-2025',
    context_salt='e164-phones',
    
    # Metrics
    detailed_metrics=True,
    max_retries=3,
    
    # Processing
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Format: {operation_s3.format}")
print(f"  Region: {operation_s3.region}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_format',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results (NEWEST file)
output_files_s3 = sorted(
    list((task_dir / 'strategy3_format' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_e164"
    
    print(f"\n📊 Sample E.164 Format Phones:")
    for _, row in result_df_s3.head(10).iterrows():
        print(f"  {row[output_field_s3]}")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and characteristics

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Phone generation rate (phones/second)
- Strategy characteristics comparison

**Note:** All strategies process 1000 phones, differences in time reflect complexity

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Multi-Country):  {elapsed_s1:6.2f}s ({1000/elapsed_s1:6.0f} phones/sec)")
print(f"  Strategy 2 (Custom Ops):     {elapsed_s2:6.2f}s ({1000/elapsed_s2:6.0f} phones/sec)")
print(f"  Strategy 3 (Format-Aware):   {elapsed_s3:6.2f}s ({1000/elapsed_s3:6.0f} phones/sec)")
print(f"  Total:                       {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📊 Strategy Characteristics:")
print(f"\n  Strategy 1 - Multi-Country:")
print(f"    • Supports US, UK, Germany, China formats")
print(f"    • Automatic country detection from data")
print(f"    • PRGN consistency for reproducibility")
print(f"    • Best for: Global datasets with regional diversity")

print(f"\n  Strategy 2 - Custom Operator Codes:")
print(f"    • Uses custom operator/area code dictionaries")
print(f"    • Mapping-based consistency (deterministic)")
print(f"    • Saves mappings for audit trail")
print(f"    • Best for: Specific operator requirements, compliance needs")

print(f"\n  Strategy 3 - Format-Aware:")
print(f"    • Standardized E.164 format")
print(f"    • Consistent formatting across all phones")
print(f"    • PRGN consistency for reproducibility")
print(f"    • Best for: API integrations, international systems")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Excludes system files automatically

**What you'll see (per strategy):**
1. **Metrics JSON**: Generation performance, country distribution, format stats
2. **Phone Mapping**: Original → Synthetic mappings (Strategy 2 only)
3. **Visualizations**: Distribution charts displayed inline (latest batch)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_multicountry', 'Strategy 1: Multi-Country'),
    ('strategy2_custom', 'Strategy 2: Custom Operator Codes'),
    ('strategy3_format', 'Strategy 3: Format-Aware')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Performance
                if 'performance' in metrics:
                    perf = metrics['performance']
                    print(f"\n   Performance:")
                    print(f"     Generation time: {perf.get('generation_time', 0):.4f}s")
                    print(f"     Records/second:  {perf.get('records_per_second', 0):,}")
                    print(f"     Memory usage:    {perf.get('memory_usage_mb', 0):.2f} MB")
                
                # Phone generator info
                if 'phone_generator' in metrics:
                    pg = metrics['phone_generator']
                    print(f"\n   Phone Generator:")
                    print(f"     Format:              {pg.get('format', 'Auto')}")
                    print(f"     Default country:     {pg.get('default_country', 'N/A')}")
                    print(f"     Preserve country:    {pg.get('preserve_country_code', False)}")
                    
                    # Country distribution
                    if 'country_distribution' in pg:
                        cd = pg['country_distribution']
                        print(f"\n     Country Distribution:")
                        print(f"       Total phones:    {cd.get('total_phones', 0):,}")
                        print(f"       Unique codes:    {cd.get('unique_country_codes', 0)}")
                        
                        top_codes = cd.get('top_country_codes', {})
                        if top_codes:
                            print(f"\n       Top Country Codes:")
                            for code, pct in list(top_codes.items())[:5]:
                                print(f"         +{code:3s} {pct:.2%}")
                
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Mapping (NEWEST) - Only for Strategy 2
    if 'strategy2' in dir_name:
        maps_dir = strategy_dir / 'maps'
        if maps_dir.exists():
            mapping_files = sorted(
                list(maps_dir.glob('*_mapping.json')),
                key=lambda x: x.stat().st_mtime, reverse=True
            )
            if mapping_files:
                print(f"\n📄 MAPPING: {mapping_files[0].name}")
                try:
                    with open(mapping_files[0], 'r') as f:
                        mapping_data = json.load(f)
                        if 'mappings' in mapping_data:
                            field = FIELD_CONFIG['strategy2_field']
                            if field in mapping_data['mappings']:
                                mappings = mapping_data['mappings'][field]
                                print(f"   Total mappings: {len(mappings)}")
                                print(f"\n   Sample Mappings (first 5):")
                                for item in mappings[:5]:
                                    orig = item.get('original', '')
                                    synth = item.get('synthetic', '')
                                    print(f"     {orig:<25} → {synth}")
                except Exception as e:
                    print(f"   ⚠️  Error: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports anonymized datasets for each strategy

**What you'll see (per strategy):**
- Available columns list
- Export path
- Preview (first 5 rows)
- Statistics (unique phones, avg length)

**Output structure:**
```
advanced_output/
├── strategy1_multicountry/
│   └── multicountry_phones.csv
├── strategy2_custom/
│   └── custom_operator_phones.csv
└── strategy3_format/
    └── e164_format_phones.csv
```

**Note:** Files include both original and generated fields for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
except NameError:
    print("❌ Run Step 4 first!")
    raise

export_base_dir = task_dir
print(f"\n📂 Export directory: {task_dir}\n")

# STRATEGY 1
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: MULTI-COUNTRY")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_multicountry'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_multicountry"
    
    if output_col_s1 in result_df_s1.columns:
        export_cols_s1 = ['record_id', field_s1, output_col_s1, 
                          'country_code', 'phone_type']
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
        
        final_df_s1 = result_df_s1[existing_cols_s1].copy()
        
        output_path_s1 = s1_dir / 'multicountry_phones.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s1.head())
        
        print(f"\n📈 Statistics:")
        print(f"   Unique original:  {result_df_s1[field_s1].nunique()}")
        print(f"   Unique synthetic: {result_df_s1[output_col_s1].nunique()}")
        print(f"   Avg length:       {result_df_s1[output_col_s1].str.len().mean():.1f} chars")
else:
    print("\n⚠️  Strategy 1 data not available")

# STRATEGY 2
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: CUSTOM OPERATOR CODES")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_custom'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s2 = f"{field_s2}_custom"
    
    if output_col_s2 in result_df_s2.columns:
        export_cols_s2 = ['record_id', field_s2, output_col_s2, 'country_code']
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
        
        final_df_s2 = result_df_s2[existing_cols_s2].copy()
        
        output_path_s2 = s2_dir / 'custom_operator_phones.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s2.head())
        
        print(f"\n📈 Statistics:")
        print(f"   Unique original:  {result_df_s2[field_s2].nunique()}")
        print(f"   Unique synthetic: {result_df_s2[output_col_s2].nunique()}")
        print(f"   Avg length:       {result_df_s2[output_col_s2].str.len().mean():.1f} chars")
else:
    print("\n⚠️  Strategy 2 data not available")

# STRATEGY 3
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: FORMAT-AWARE")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_format'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s3 = f"{field_s3}_e164"
    
    if output_col_s3 in result_df_s3.columns:
        export_cols_s3 = ['record_id', field_s3, output_col_s3, 'country_code']
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
        
        final_df_s3 = result_df_s3[existing_cols_s3].copy()
        
        output_path_s3 = s3_dir / 'e164_format_phones.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s3.head())
        
        print(f"\n📈 Statistics:")
        print(f"   Unique original:  {result_df_s3[field_s3].nunique()}")
        print(f"   Unique synthetic: {result_df_s3[output_col_s3].nunique()}")
        print(f"   Avg length:       {result_df_s3[output_col_s3].str.len().mean():.1f} chars")
else:
    print("\n⚠️  Strategy 3 data not available")

print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 strategies implemented and compared
- ✅ Multi-country phone generation (US, UK, Germany, China)
- ✅ Custom operator code dictionaries
- ✅ Format-aware generation (E.164)
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Multi-country strategy provides geographic diversity
- Custom operator codes ensure specific area/carrier codes
- Format-aware generation standardizes output for APIs
- PRGN vs mapping: different consistency mechanisms for different needs
- All strategies preserve country code characteristics

**Strategy recommendations:**
- **Use Strategy 1** when: Global datasets, regional diversity needed
- **Use Strategy 2** when: Specific operator requirements, compliance needs
- **Use Strategy 3** when: API integrations, standardized formats required

**Next steps:**
- Test with your own datasets
- Customize operator dictionaries for your region
- Combine strategies for complex scenarios
- Deploy to production pipelines

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)
- 📞 [Phone Generator Guide](../../docs/generators/phone_generator.md)
- 🔒 [PRGN Consistency Guide](../../docs/prgn_mechanism.md)